# rPPG — 5. tur: telefon verisi (MMPD, VIPL-HR) + DLCN (gece ışığı) + COHFACE (Kaggle)

3. tur modelinden (temiz etiket; telefon/USB testinde en iyisi) başlayıp yeni veriyle ince ayar yapar:

* **MMPD (mini)**: 33 kişi × 20 video, **Samsung Galaxy S22 Ultra** telefonu; 4 ışık, 4 hareket durumu.
* **VIPL-HR**: 107 kişi × 9 senaryo; webcam, RealSense ve **Huawei P9 telefonu** (kızılötesi kaynak 3 kullanılmaz).
* **COHFACE**: 40 kişi × 4 video, yoğun sıkıştırılmış webcam (stüdyo ve doğal ışık).
* **DLCN**: 98 kişi × 8 video, gece ve değişen ışık (10–100 lux), dinlenme + egzersiz; Logitech C922, CMS50E.
  Kaggle veri seti olarak eklenir, indirme beklemez.
* Önceki veri: MCD-rPPG, UBFC-rPPG, PURE (temiz etiket filtresiyle).

Her yeni veri setinde numarası 5'e bölünen kişiler **test** klasörüne ayrılır, eğitime hiç girmez. Sonda modeller
ana testte (28 video) ve **telefon testlerinde** (MMPD, VIPL telefon) yan yana ölçülür.

**Gerekenler:**
0. **Add Input** → arama: `rppg-dlcn` → **dalaoplan/rppg-dlcn** veri setini ekleyin (Hugging Face onayı gerekmez).
1. Hugging Face hesabıyla şu üç sayfada erişim isteği gönderilmiş ve **onaylanmış** olmalı:
   [mini_MMPD](https://huggingface.co/datasets/WeiQian98/mini_MMPD),
   [VIPL-HR](https://huggingface.co/datasets/WeiQian98/VIPL-HR),
   [COHFACE](https://huggingface.co/datasets/WeiQian98/COHFACE).
2. Kaggle **Add-ons → Secrets** altında `HF_TOKEN` bu not defterine bağlı olmalı.
3. **Settings:** Accelerator **GPU T4 x2**, **Turn on internet**.

DLCN eklenmemişse ya da erişimi onaylanmamış veri seti atlanır, eğitim diğer verilerle yine yapılır. 2. hücre her veri setinden bir videoyu
hemen işleyip okumanın çalıştığını gösterir; orada hata görürseniz çalıştırmayı durdurup haber verin.

**Çalıştırma:** **Save Version → Save & Run All (Commit)**; ≈11 saat. Bitince **Output** → `cikti` klasörünü indirin.

In [ ]:
# 1) Ortam, kod ve rPPG-Toolbox
import os, subprocess, shutil, sys, time, glob, json
from datetime import datetime, timedelta

T0 = datetime.now()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert shutil.which("nvidia-smi"), "GPU yok: Settings -> Accelerator: GPU T4 x2"
assert subprocess.run(["curl", "-sI", "https://github.com"], capture_output=True).returncode == 0, \
    "İnternet kapalı: Settings -> Turn on internet"

REPO = "/kaggle/working/rppg"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", REPO], check=True)
TB = os.path.join(REPO, "external", "rPPG-Toolbox")
if not os.path.exists(TB):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ubicomplab/rPPG-Toolbox.git", TB], check=True)
os.chdir(REPO)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN var")
except Exception:
    print("!" * 70)
    print("UYARI: HF_TOKEN yok, MMPD / VIPL-HR / COHFACE atlanacak. Add-ons -> Secrets -> HF_TOKEN kutusunu "
          "işaretleyin ve not defterini yeniden başlatın.")
    print("!" * 70)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# 2) Erişim ve okuma denemesi: her yeni veri setinden bir video (birkaç dakika)
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")
r = subprocess.run([PY, "-u", "scripts/dl_prepare.py", "--out", "/tmp/dl_deneme", "--raw", "/tmp/dl_deneme_raw",
                    "--dlcn", "1", "--cohface", "1", "--mmpd", "1", "--vipl", "1", "--workers", "1"],
                   cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-4000:], r.stderr[-3000:])
print("işlenen:", sorted(os.path.basename(p) for p in glob.glob("/tmp/dl_deneme/*/*.npz")))

In [ ]:
# 3) Klasörler, süre, başlangıç modeli
DATA, RAW = "/tmp/dl", "/tmp/dl_raw"
OUT = "/kaggle/working/cikti"
LOGS = os.path.join(OUT, "gunlukler")
for d in (DATA, RAW, OUT, LOGS):
    os.makedirs(d, exist_ok=True)

TRAIN_UNTIL = T0 + timedelta(hours=12 - 1.8)          # Kaggle oturum sınırı 12 saat; değerlendirme payı
INIT = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur3/last.pth")   # 3. tur, son epoch
TUR1 = os.path.join(REPO, "results/derin_ogrenme/kaggle/cikti/best.pth")
assert os.path.exists(INIT), INIT
print(f"/tmp boş {shutil.disk_usage('/tmp').free / 1e9:.0f} GB | eğitim en geç {TRAIN_UNTIL:%H:%M}")

In [ ]:
# 4) Veri hazırlama (arka planda, indir-işle-sil)
#    A: MCD-rPPG 600 kişi   B: test kümesi -> UBFC 34 -> PURE 60   C: DLCN (Kaggle girdisi)
#    D: COHFACE -> MMPD -> VIPL-HR (Hugging Face; test kişileri önce)
def launch(name, cmd):
    return subprocess.Popen(["bash", "-c", cmd], cwd=REPO, env=env,
                            stdout=open(os.path.join(LOGS, name + ".log"), "a"), stderr=subprocess.STDOUT)

P = f"{PY} -u scripts/dl_prepare.py --out {DATA} --raw {RAW}"
jobs = {
    "hazirla_mcd": launch("hazirla_mcd", f"{P} --mcd 600 --workers 2 --max-sec 45"),
    "hazirla_test_ubfc_pure": launch("hazirla_test_ubfc_pure",
        f"{P} --test-download --workers 2; {P} --ubfc 34 --workers 1; {P} --pure 60 --workers 1"),
    "hazirla_dlcn": launch("hazirla_dlcn", f"{P} --dlcn 784 --workers 2"),
    "hazirla_yeni": launch("hazirla_yeni",
        f"{P} --cohface 164 --workers 2; {P} --mmpd 660 --workers 2; {P} --vipl 2500 --workers 2 --max-sec 60"),
}
print("veri hazırlama başladı")

In [ ]:
# 5) Eğitim: 300 video hazır olunca başlar; yeni gelenler her epoch başında eklenir
SUBS = ("mcd", "ubfc", "pure", "dlcn", "mmpd", "vipl", "cohface")
TESTS = ("test", "test_mmpd", "test_vipl", "test_dlcn", "test_cohface")
def counts(subs=SUBS):
    return {s: len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in subs}

while sum(counts().values()) < 300 and any(j.poll() is None for j in jobs.values()):
    print(time.strftime("%H:%M"), counts(), flush=True)
    time.sleep(120)

train = subprocess.Popen(
    [PY, "-u", "scripts/dl_train.py", "--data", DATA, "--out", OUT, "--init", INIT, "--epochs", "100",
     "--steps", "500", "--lr", "1e-4", "--deadline", TRAIN_UNTIL.strftime("%Y-%m-%d %H:%M")],
    cwd=REPO, env=env, stdout=open(os.path.join(LOGS, "egitim.log"), "a"), stderr=subprocess.STDOUT)
print("eğitim başladı")

In [ ]:
# 6) İzleme: 10 dakikada bir özet
def tail(path, n):
    try:
        return open(path, encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

while train.poll() is None:
    ep = [l for l in tail(os.path.join(LOGS, "egitim.log"), 400) if l.startswith("epoch ")]
    print(f"{time.strftime('%H:%M')} | {counts()} | test {counts(TESTS)} | disk {shutil.disk_usage('/tmp').free/1e9:.0f} GB"
          f" | {ep[-1] if ep else 'ilk epoch sürüyor'}", flush=True)
    time.sleep(600)
print("\n".join(tail(os.path.join(LOGS, "egitim.log"), 15)))

In [ ]:
# 7) Veri kalitesi özeti (eğitimde kullanılan filtre; kalite.csv çıktıya kopyalanır)
r = subprocess.run([PY, "scripts/dl_quality.py", "--data", DATA, "--workers", "4", "--sources", *SUBS],
                   cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-3000:], r.stderr[-2000:])
if os.path.exists(os.path.join(DATA, "kalite.csv")):
    shutil.copy(os.path.join(DATA, "kalite.csv"), OUT)

In [ ]:
# 8) Değerlendirme: ana test (28 video) + telefon testleri (MMPD, VIPL) + DLCN + COHFACE
for j in jobs.values():
    if j.poll() is None:
        j.terminate()
print("test videoları:", counts(TESTS))
for t in TESTS:
    d = os.path.join(DATA, t)
    if not glob.glob(os.path.join(d, "*.npz")):
        continue
    out = OUT if t == "test" else os.path.join(OUT, t)
    r = subprocess.run([PY, "scripts/dl_evaluate.py", "--model", os.path.join(OUT, "best.pth"), "--test", d,
                        "--data", DATA, "--out", out, "--history", os.path.join(OUT, "gecmis.json"),
                        "--compare", f"3. tur (başlangıç)={INIT}", f"1. tur={TUR1}"],
                       cwd=REPO, capture_output=True, text=True, env=env)
    print(f"===== {t}\n", r.stdout[-5000:], r.stderr[-2000:])

In [ ]:
# 9) Çıktılar
for p in sorted(glob.glob(os.path.join(OUT, "**"), recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}")
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın
print("süre:", datetime.now() - T0)